In [1]:
#kernel thesis thesis_unishape
#https://github.com/ZLiu21/UniShape/blob/main/unishape_zeroshot.py
import pickle
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import shuffle
import numpy as np
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split

In [3]:
df = pd.read_pickle("screw_data_s02-v2_identical-to-v1.pkl")
#df.head()

In [4]:
x_data = np.array(df['torque_values'].tolist())[..., np.newaxis, :]  
y_data = np.array(df['class_values'].tolist())

le = LabelEncoder()
y_encoded = le.fit_transform(y_data)

x_data = np.transpose(x_data, (0, 2, 1))  

X_train, X_test, y_train, y_test = train_test_split(x_data, y_encoded, test_size=0.2, stratify=y_encoded, random_state=42)



In [5]:
def write_ts(filename, X, y, class_labels):
    lines = []
    lines.append("@problemName Torque\n")
    lines.append("@timestamps false\n")
    lines.append("@univariate true\n")
    lines.append("@equalLength true\n")
    lines.append("@seriesLength 800\n")
    lines.append("@classLabel true " + " ".join(map(str,class_labels)) +"\n")
    lines.append("@data\n")
    for xi, yi in zip(X, y):
        xi = xi.detach().cpu().numpy()
        series = ",".join(xi.reshape(-1).astype(str))
        lines.append(f"{series}:{int(yi)}\n")
    with open(filename, "w") as f:
        f.writelines(lines)

In [6]:
base_path = "./UniShape/30NewDatasets/Torque"
import os
os.makedirs(base_path, exist_ok=True)

In [7]:
class_labels = np.unique(y_encoded)

In [7]:


#write_ts(os.path.join(base_path, "Torque_TRAIN.ts"),X_train,y_train,class_labels)

#write_ts(os.path.join(base_path, "Torque_TEST.ts"),X_test,y_test,class_labels)

In [8]:
import sys
import torch
sys.path.append(os.path.abspath("./UniShape"))
from models.unishapemodel_zeroshot import UniShapeModel
from utils.util import load_pretrained_model

/opt/conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/conda/lib/python3.11/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [10]:
import os
import sys

sys.path.append(os.path.abspath("./UniShape"))

import argparse
import numpy as np
import torch
from torch.utils.data import DataLoader
from utils.util import set_seed, build_30new_dataset_raw_split, load_pretrained_model, New30Dataset, getInteSet
from sklearn.preprocessing import LabelEncoder
from models.unishapemodel_zeroshot import UniShapeModel
from sklearn.metrics import accuracy_score, f1_score
from sklearn.ensemble import RandomForestClassifier


def zero_feature_extraction(val_loader, models):
    val_pred_embeds = []
    real_labels = []

    for data, target in val_loader:
        with torch.no_grad():
            val_pred = models[0](data, target)
            val_pred_embeds.append(val_pred.cpu().numpy())
            real_labels.append(target.cpu().numpy())

    val_pred_embeds = np.concatenate(val_pred_embeds)
    real_labels = np.concatenate(real_labels)
    return val_pred_embeds, real_labels


uni_30new_ts_sets = {"Tools", "SharePriceIncrease", "ShakeGestureWiimoteZ_eq", "PLAID_eq", "PickupGestureWiimoteZ_eq", "PhoneHeartbeatSound", "MelbournePedestrian_nmv", "KeplerLightCurves",
                  "GesturePebbleZ2_eq", "GesturePebbleZ1_eq", "GestureMidAirD3_eq", "GestureMidAirD2_eq", "GestureMidAirD1_eq", "FloodModeling3_disc", "FloodModeling2_disc", "FloodModeling1_disc",
                  "ElectricDeviceDetection", "DodgerLoopWeekend_nmv", "DodgerLoopGame_nmv", "DodgerLoopDay_nmv", "Covid3Month_disc", "Colposcopy", "AsphaltRegularityUni_eq", "AsphaltPavementTypeUni_eq",
                  "AsphaltObstaclesUni_eq", "AllGestureWiimoteZ_eq", "AllGestureWiimoteY_eq", "AllGestureWiimoteX_eq", "AconityMINIPrinterSmall_eq", "AconityMINIPrinterLarge_eq"}

if __name__ == '__main__': 
    parser = argparse.ArgumentParser()
    # Base setup
    parser.add_argument('--random_seed', type=int, default=42, help='shuffle seed')

    # Dataset setup  
    parser.add_argument('--dataset', type=str, default="Torque", help='dataset(in ucr)')  # "Tools"
    parser.add_argument('--dataroot', type=str, default='./30NewDatasets', help='path of UCR folder')
    parser.add_argument('--num_class', type=int, default=0, help='number of class')
    
    # Model
    parser.add_argument('--in_channels', type=int, default=128)
    parser.add_argument('--window_emb_dim', type=int, default=128)
    parser.add_argument('--window_size', type=int, default=16)
    parser.add_argument('--stride', type=int, default=16)
    parser.add_argument('--scale_len', type=int, default=4)

    # training setup
    parser.add_argument('--loss', type=str, default='cross_entropy', help='loss function')
    parser.add_argument('--optimizer', type=str, default='adam', help='optimizer')
    parser.add_argument('--lr', type=float, default=0.001, help='learning rate')
    parser.add_argument('--weight_decay', type=float, default=0.0, help='weight decay')
    parser.add_argument('--batch_size', type=int, default=512, help='')
    parser.add_argument('--epoch', type=int, default=300, help='training epoch')
    parser.add_argument('--cuda', type=str, default='cuda:0')

    args = parser.parse_args(args=[])

    device = torch.device(args.cuda if torch.cuda.is_available() else "cpu")
    set_seed(args)
    args.dataset = "Torque"
    args.dataroot = "./UniShape/30NewDatasets"
    train_x, train_target, test_x, test_target, num_classes = build_30new_dataset_raw_split(args)
    train_x = train_x.transpose(0, 2, 1)
    test_x = test_x.transpose(0, 2, 1)
  
    args.num_class = num_classes
    args.seq_len = train_x.shape[1]
    args.model_series_size = train_x.shape[1]
    args.input_size = train_x.shape[2]
    args.model_series_size = 512
    args.seq_len = 512
    args.out_channels = num_classes
    args.load_checkpoint_path = "./UniShape/pretrained_model_ckpt/unishape_checkpoint_zeroshot.pth"
    
    
    model_list = []
    
    for _ in range(1):  
    
        model = UniShapeModel( config=args,
            series_size=args.model_series_size, in_channels=args.in_channels, window_emb_dim=args.window_emb_dim,
            out_channels=args.out_channels, window_size=args.window_size, stride=args.stride, shape_alpha=0.01, shape_sparse_ratio=0.6,
            scale_len=args.scale_len
        )
        
        model = load_pretrained_model(args, model)
        model = model.to(device)
        model_list.append(model)
  
    print('Start zero shot feature extraction and evaluate: ')
   
    train_dataset = getInteSet(train_x.transpose(0, 2, 1))
    train_dataset =  np.squeeze(train_dataset, axis=2).transpose(0, 2, 1) 
    test_dataset = getInteSet(test_x.transpose(0, 2, 1))
    test_dataset =  np.squeeze(test_dataset, axis=2).transpose(0, 2, 1)
    
    le = LabelEncoder()
    train_target = le.fit_transform(train_target)
    test_target = le.transform(test_target)

    train_set = New30Dataset(torch.from_numpy(train_dataset).type(torch.FloatTensor).to(device),
                           torch.from_numpy(train_target).type(torch.FloatTensor).to(device).to(torch.int64))
    test_set = New30Dataset(torch.from_numpy(test_dataset).type(torch.FloatTensor).to(device),
                          torch.from_numpy(test_target).type(torch.FloatTensor).to(device).to(torch.int64))

    train_loader = DataLoader(train_set, batch_size=args.batch_size, num_workers=0, drop_last=False)
    test_loader = DataLoader(test_set, batch_size=args.batch_size, num_workers=0)
 
    train_embeds, train_labels = zero_feature_extraction(train_loader, model_list)
    test_embeds, test_labels = zero_feature_extraction(test_loader, model_list)
    
    predictor = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=0)
    predictor.fit(train_embeds, train_labels)
    
    y_pred = predictor.predict(test_embeds)
    test_f1 = f1_score(test_labels, y_pred, average='macro')
    
    print("F1 Macro = ", test_f1)
    print('Done!')

/workspace/UniShape/utils/util.py:58: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(args.load_checkpoint_path, map_location=f'cuda:0')


Start zero shot feature extraction and evaluate: 
F1 Macro =  0.29136094423146447
Done!


## 3-cv Unishape

In [ ]:
import os
import sys

sys.path.append(os.path.abspath("./UniShape"))

import argparse
import numpy as np
import torch
from torch.utils.data import DataLoader
from utils.util import set_seed, build_30new_dataset_raw_split, load_pretrained_model, New30Dataset, getInteSet
from sklearn.preprocessing import LabelEncoder
from models.unishapemodel_zeroshot import UniShapeModel
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score


def zero_feature_extraction(val_loader, models):
    val_pred_embeds = []
    real_labels = []

    for data, target in val_loader:
        with torch.no_grad():
            val_pred = models[0](data, target)
            val_pred_embeds.append(val_pred.cpu().numpy())
            real_labels.append(target.cpu().numpy())

    val_pred_embeds = np.concatenate(val_pred_embeds)
    real_labels = np.concatenate(real_labels)
    return val_pred_embeds, real_labels


uni_30new_ts_sets = {"Tools", "SharePriceIncrease", "ShakeGestureWiimoteZ_eq", "PLAID_eq", "PickupGestureWiimoteZ_eq", "PhoneHeartbeatSound", "MelbournePedestrian_nmv", "KeplerLightCurves",
                  "GesturePebbleZ2_eq", "GesturePebbleZ1_eq", "GestureMidAirD3_eq", "GestureMidAirD2_eq", "GestureMidAirD1_eq", "FloodModeling3_disc", "FloodModeling2_disc", "FloodModeling1_disc",
                  "ElectricDeviceDetection", "DodgerLoopWeekend_nmv", "DodgerLoopGame_nmv", "DodgerLoopDay_nmv", "Covid3Month_disc", "Colposcopy", "AsphaltRegularityUni_eq", "AsphaltPavementTypeUni_eq",
                  "AsphaltObstaclesUni_eq", "AllGestureWiimoteZ_eq", "AllGestureWiimoteY_eq", "AllGestureWiimoteX_eq", "AconityMINIPrinterSmall_eq", "AconityMINIPrinterLarge_eq"}

if __name__ == '__main__': 
    parser = argparse.ArgumentParser()
    parser.add_argument('--random_seed', type=int, default=42, help='shuffle seed')
    parser.add_argument('--dataset', type=str, default="Torque", help='dataset(in ucr)')  # "Tools"
    parser.add_argument('--dataroot', type=str, default='./30NewDatasets', help='path of UCR folder')
    parser.add_argument('--num_class', type=int, default=0, help='number of class')
    parser.add_argument('--in_channels', type=int, default=128)
    parser.add_argument('--window_emb_dim', type=int, default=128)
    parser.add_argument('--window_size', type=int, default=16)
    parser.add_argument('--stride', type=int, default=16)
    parser.add_argument('--scale_len', type=int, default=4)
    parser.add_argument('--loss', type=str, default='cross_entropy', help='loss function')
    parser.add_argument('--optimizer', type=str, default='adam', help='optimizer')
    parser.add_argument('--lr', type=float, default=0.001, help='learning rate')
    parser.add_argument('--weight_decay', type=float, default=0.0, help='weight decay')
    parser.add_argument('--batch_size', type=int, default=512, help='')
    parser.add_argument('--epoch', type=int, default=300, help='training epoch')
    parser.add_argument('--cuda', type=str, default='cuda:0')

    args = parser.parse_args(args=[])

    device = torch.device(args.cuda if torch.cuda.is_available() else "cpu")
    set_seed(args)
    args.dataset = "Torque"
    args.dataroot = "./UniShape/30NewDatasets"
    train_x, train_target, test_x, test_target, num_classes = build_30new_dataset_raw_split(args)
    train_x = train_x.transpose(0, 2, 1)
    test_x = test_x.transpose(0, 2, 1)
  
    args.num_class = num_classes
    args.seq_len = train_x.shape[1]
    args.model_series_size = train_x.shape[1]
    args.input_size = train_x.shape[2]
    args.model_series_size = 512
    args.seq_len = 512
    args.out_channels = num_classes
    args.load_checkpoint_path = "./UniShape/pretrained_model_ckpt/unishape_checkpoint_zeroshot.pth"
    
    
    model_list = []
    
    for _ in range(1):  
    
        model = UniShapeModel( config=args,
            series_size=args.model_series_size, in_channels=args.in_channels, window_emb_dim=args.window_emb_dim,
            out_channels=args.out_channels, window_size=args.window_size, stride=args.stride, shape_alpha=0.01, shape_sparse_ratio=0.6,
            scale_len=args.scale_len
        )
        
        model = load_pretrained_model(args, model)
        model = model.to(device)
        model_list.append(model)
  
    print('Start zero shot feature extraction and evaluate: ')
   
    train_dataset = getInteSet(train_x.transpose(0, 2, 1))
    train_dataset =  np.squeeze(train_dataset, axis=2).transpose(0, 2, 1) 
    test_dataset = getInteSet(test_x.transpose(0, 2, 1))
    test_dataset =  np.squeeze(test_dataset, axis=2).transpose(0, 2, 1)
    
    le = LabelEncoder()
    train_target = le.fit_transform(train_target)
    test_target = le.transform(test_target)

    train_set = New30Dataset(torch.from_numpy(train_dataset).type(torch.FloatTensor).to(device),
                           torch.from_numpy(train_target).type(torch.FloatTensor).to(device).to(torch.int64))
    test_set = New30Dataset(torch.from_numpy(test_dataset).type(torch.FloatTensor).to(device),
                          torch.from_numpy(test_target).type(torch.FloatTensor).to(device).to(torch.int64))

    train_loader = DataLoader(train_set, batch_size=args.batch_size, num_workers=0, drop_last=False)
    test_loader = DataLoader(test_set, batch_size=args.batch_size, num_workers=0)
 
    train_embeds, train_labels = zero_feature_extraction(train_loader, model_list)
    test_embeds, test_labels = zero_feature_extraction(test_loader, model_list)
    
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    cv_scores = []

    for train_idx, val_idx in skf.split(train_embeds, train_labels):
        X_train, X_val = train_embeds[train_idx], train_embeds[val_idx]
        y_train, y_val = train_labels[train_idx], train_labels[val_idx]
        clf = RandomForestClassifier(n_estimators=200, random_state=0)
        clf.fit(X_train, y_train)
        preds= clf.predict(X_val)
        f1=f1_score(y_val, preds, average="macro")

        cv_scores.append(f1)

    print(np.mean(cv_scores),np.std(cv_scores))

C:\Users\Patrick\UniShape\utils\util.py:58: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(args.load_checkpoint_path, map_location=f'cuda:0')


Start zero shot feature extraction and evaluate: 
0.2849200438479662 0.006260140869010086


In [ ]:
final_clf = clf

test_pred = final_clf.predict(test_embeds)
test_f1 = f1_score(test_labels, test_pred, average="macro")

print(f"test f1 score {test_f1}")

In [5]:
final_clf = RandomForestClassifier(n_estimators=200, random_state=0)
final_clf.fit(train_embeds, train_labels)

test_pred = final_clf.predict(test_embeds)
test_f1 = f1_score(test_labels, test_pred, average="macro")

print(test_f1)

0.29136094423146447
